`%reload_ext` loads the extension on first run and reloads it on subsequent runs, so re-running this cell always picks up any changes to the autoreload extension.

In [1]:
%reload_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import build123d as bd
from build123d import (
    Box,
    Compound,
    Cone,
    Cylinder,
    Part,
    Pos,
    ShapeList,
    export_stl,
    extrude,
)
from ocp_vscode import (
    get_defaults,
    reset_show,
    set_defaults,
    set_port,
    show,
    show_object,
)

import print_scripts_tree_d.shapes as shapes
from print_scripts_tree_d.export import save_stl

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("build123d").setLevel(logging.WARNING)
models = Path("models")
models.mkdir(exist_ok=True)
reset_show()
set_port(3939)

## Box with Cylinder Clip

In [2]:
from typing import cast

from build123d import Compound, Pos, Rot, ShapeList

from print_scripts_tree_d.params import CylinderClipParams, RoundedBoxParams

cp = CylinderClipParams()

bp = RoundedBoxParams(
    length=90.0,
    width=15.0,
    height=6.0,
    wall_thickness=4.0,
    corner_radius=2.5,
    top_fillet_radius=5.0,
    bottom_fillet_radius=1.0,
)
box = shapes.make_rounded_box(
    length=bp.length,
    width=bp.width,
    height=bp.height,
    wall_thickness=bp.wall_thickness,
    corner_radius=bp.corner_radius,
    top_fillet_radius=bp.top_fillet_radius,
    bottom_fillet_radius=bp.bottom_fillet_radius,
)
show(box)

Filleting top rim r=1.900...
Filleting bottom rim r=1.000...


+


In [9]:
clip = shapes.make_cylinder_clip(
    bore_diameter=cp.bore_diameter,
    body_depth=cp.body_depth,
    wall_thickness=cp.wall_thickness,
    flange_overlap=cp.flange_overlap,
    flange_thickness=cp.flange_thickness,
    tab_count=cp.tab_count,
    tab_protrusion=cp.tab_protrusion,
    tab_length=cp.tab_length,
    tab_width=cp.tab_width,
    slot_width=cp.slot_width,
    clearance=cp.clearance,
    flat_bottom=cp.flat_bottom,
    flat_fillet_r=1,
    flat_inner_margin=-0.6,
)
show(clip)

Building clip body OD=10.6 ID=8.2 depth=10.0...
Cutting 4 spring-finger slots...
Adding 4 snap tabs...


c


In [11]:
tx = bp.length / 2 + cp.body_depth / 2
clip_rotated = Rot(0, 90, 0) * clip
# Derive z_offset from the actual clip geometry so it stays correct
# regardless of which flat_inner_margin was used to build the clip.
flat_bottom_z = clip_rotated.bounding_box().min.Z
z_offset = -flat_bottom_z - bp.height / 2
clip_at_end = Pos(tx, 0, z_offset) * clip_rotated

result = box + clip_at_end
if isinstance(result, ShapeList):
    assembly = Compound(children=list(result))
else:
    assembly = cast(Compound, result)

# Lift so both flat faces sit on Z = 0.
assembly = cast(Compound, Pos(0, 0, bp.height / 2) * assembly)

reset_show()
show(assembly)

+


In [ ]:
save_stl(assembly, str(models / "box_with_clip.stl"))

## Column

In [7]:
col_body = Cylinder(1.5, 50)
col_foot = Cone(bottom_radius=1, top_radius=1.5, height=3)
column = shapes.make_column(
    body=col_body,
    height=50,
    foot=col_foot,
    diameter=3,
    gusset_size=3,
    gusset_thickness=1.5,
    gusset_position_z="top",
    gusset_orientation_xy=(0,120,240)
)
show(column)

Adding 3 gussets at top...


+


## Table top

In [ ]:
table_top = shapes.make_hexagonal_mesh(
    length=6,
    width=6,
    thickness=1,
    hex_radius=4,
    spacing=1,
    outer_border=1,
)
show(table_top)

## Table assembly

In [ ]:
table = shapes.make_table(
    table_top=table_top,
    columns=[column.rotate(angle=180, axis=bd.Axis.X)],
    column_positions=[(50, 50)],
)
reset_show()
show(table)

In [ ]:
save_stl(table, str(models / "table.stl"))

## Rounded Box

In [2]:
from print_scripts_tree_d.params import RoundedBoxParams

p = RoundedBoxParams()
box = shapes.make_rounded_box(
    length=p.length,
    width=p.width,
    height=p.height,
    wall_thickness=p.wall_thickness,
    corner_radius=p.corner_radius,
    top_fillet_radius=0.8,
    bottom_fillet_radius=0.5,
)
reset_show()
show(box)

Filleting top rim r=0.800...
Filleting bottom rim r=0.500...


+


In [ ]:
save_stl(box, str(models / "rounded_box.stl"))